# Narrative Miners: Uncover the Stories That Drive Markets

## Automated Analysis of Market Narratives Across Multiple Document Sources

## Why It Matters

Understanding how market narratives emerge and evolve across different information sources is crucial for investment decision-making, but manually tracking narrative development across scattered news coverage, earnings calls, and regulatory filings is time consuming. Investment decisions need systematic analysis of narrative progression to identify emerging trends and timing patterns.

## What It Does

The `NarrativeMiner` class in the bigdata-research-tools package systematically tracks narrative evolution across multiple document types using unstructured data from news, transcripts, and filings. Built for analysts, portfolio managers, and investment professionals, it transforms scattered narrative signals into quantified trend intelligence and identifies timing patterns across different information sources.

## How It Works

The `NarrativeMiner` combines **multi-source content retrieval**, **temporal narrative tracking**, and **cross-source comparative analysis** to deliver:

- **Cross-document narrative mapping** across news media, earnings calls, and SEC filings
- **Temporal evolution tracking** showing how narratives develop and change over time across sources
- **Intensity measurement** quantifying narrative prevalence and significance across document types

## A Real-World Use Case

This cookbook demonstrates the complete workflow through analyzing "AI Bubble Concerns" narrative as it emerges and evolves across news, earnings calls, and regulatory filings, highlighting the difference between public discourse and corporate communications.

## Setup and Imports

## Async Compatibility Setup

**Run this cell first** - Required for Google Colab, Jupyter Notebooks, and VS Code with Jupyter extension:

### Why is this needed?

Interactive environments (Colab, Jupyter) already have an asyncio event loop running. When bigdata-research-tools makes async API calls (like to OpenAI), you'll get this error without nest_asyncio:

```
RuntimeError: asyncio.run() cannot be called from a running event loop
```

The `nest_asyncio.apply()` command patches this to allow nested event loops.

💡 **Tip**: If you're unsure which environment you're in, just run the cell below - it won't hurt in any environment!

In [1]:
import datetime
start = datetime.datetime.now()

try:
    import asyncio
    asyncio.get_running_loop()
    import nest_asyncio; nest_asyncio.apply()
    print("✅ nest_asyncio applied")
except (RuntimeError, ImportError):
    print("✅ nest_asyncio not needed or not available")

✅ nest_asyncio applied


## Environment Setup

The following cell configures the necessary path for the analysis

In [2]:
import os
import sys

current_dir = os.getcwd()
if current_dir not in sys.path:
    sys.path.append(current_dir)
print(f"✅ Local environment setup complete")

✅ Local environment setup complete


## Import Required Libraries

Import the core libraries needed for narrative mining analysis, including the custom visualization and analysis tools.

In [3]:
from IPython.display import display, HTML, IFrame
import pandas as pd
from pandas import merge

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from datetime import datetime
import warnings

from src.tool import (
    load_results, 
    extract_narrative_insights,
    create_source_summary,
    display_sample_data,
    visualize_cross_source_narratives, 
    visualize_news_narrative_breakdown
)

from bigdata_research_tools.workflows.narrative_miner import NarrativeMiner
from bigdata_research_tools.excel import ExcelManager
from bigdata_client import Bigdata
from bigdata_client.daterange import RollingDateRange
from bigdata_client.models.sources import Source
from bigdata_client.models.search import DocumentType

## Optional: Plotly Display Configuration

For better visualization rendering, you can also set the Plotly renderer:

In [4]:
import plotly
import plotly.graph_objects as go
import plotly.io as pio

# Try to detect the environment and set appropriate renderer
try:
    # Check if we're in JupyterLab
    import os
    if 'JUPYTERHUB_SERVICE_PREFIX' in os.environ or 'JPY_SESSION_NAME' in os.environ:
        pio.renderers.default = 'jupyterlab'
        print("✅ Plotly configured for JupyterLab")
    else:
        # Default for VS Code, Jupyter Notebook, etc.
        pio.renderers.default = 'plotly_mimetype+notebook'
        print("✅ Plotly configured for Jupyter/VS Code")
except:
    # Fallback to a more universal renderer
    pio.renderers.default = 'notebook'
    print("✅ Plotly configured with fallback renderer")



✅ Plotly configured for Jupyter/VS Code


/opt/anaconda/envs/pricing_power/lib/python3.11/site-packages/kaleido/__init__.py:14: UserWarning:




This means that static image generation (e.g. `fig.write_image()`) will not work.

Please upgrade Plotly to version 6.1.1 or greater, or downgrade Kaleido to version 0.2.1.




## Define Output Paths

We define the output paths for our narrative mining results.

In [5]:
# Define output file paths for our results
output_dir = "output"
os.makedirs(output_dir, exist_ok=True)

news_results_path = f"{output_dir}/ai_bubble_news.xlsx"
visualization_path = f"{output_dir}/ai_bubble_narratives.html"

## Load Credentials

In [6]:
from dotenv import load_dotenv
from pathlib import Path

script_dir = Path(__file__).parent if '__file__' in globals() else Path.cwd()
load_dotenv(script_dir / '.env')

BIGDATA_USERNAME = os.getenv('BIGDATA_USERNAME')
BIGDATA_PASSWORD = os.getenv('BIGDATA_PASSWORD')
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')

if not all([BIGDATA_USERNAME, BIGDATA_PASSWORD, OPENAI_API_KEY]):
    print("❌ Missing required environment variables")
    raise ValueError("Missing required environment variables. Check your .env file.")
else:
    print("✅ Credentials loaded from .env file")

✅ Credentials loaded from .env file


## Connecting to Bigdata

Create a Bigdata object with your credentials.

In [7]:
bigdata = Bigdata(BIGDATA_USERNAME, BIGDATA_PASSWORD)

## Defining the Narrative Analysis Parameters

### Fixed Parameters
- **AI Bubble Narratives** (`main_narratives`): Specific narrative sentences related to AI bubble concerns
- **Common Parameters** (`common_params`): Shared configuration across all narrative miners
- **Model Selection** (`llm_model`): The LLM model used for narrative labeling and analysis
- **Time Period** (`start_date` and `end_date`): The date range over which to run the analysis
- **Rerank Threshold** (`rerank_threshold`): Cross-encoder threshold for result relevance filtering
- **Document Limits** (`document_limit`): Maximum number of documents to retrieve per query
- **Search Frequency** (`freq`): Frequency of date ranges for search operations

In [9]:
# AI Bubble Narratives
main_narratives = ['A proposed tax impacting companies employing H-1B visa holders.',
 'New tax may hinder recruitment of skilled H-1B visa workers.',
 'Higher costs may deter companies from hiring H-1B visa candidates.',
 'Companies may prioritize domestic candidates over H-1B visa applicants.',
 'Tax may exacerbate shortages in specialized skills from H-1B visa workers.',
 'Companies may alter recruitment strategies to minimize H-1B visa reliance.',
 'Tax increases operational costs for companies employing H-1B visa workers.',
 'Companies may need to allocate budgets for new H-1B visa taxes.',
 'Increased costs from H-1B visa taxes may reduce profit margins.',
 'Companies may reassess the financial viability of hiring H-1B visa workers.',
 'Companies may restructure operations to offset H-1B visa tax costs.',
 "Tax may affect companies' competitiveness in the global market.",
 'Higher costs may weaken competitiveness against international firms not facing H-1B visa taxes.',
 'Companies may need to adjust pricing due to increased H-1B visa costs.',
 'Competitors may attract H-1B visa talent by offering better conditions.',
 'Companies may need to rethink global strategies to attract H-1B visa talent.',
 'Tax influences long-term workforce planning regarding H-1B visa employees.',
 'Companies may struggle to develop a robust talent pipeline for H-1B visa roles.',
 'Tax may influence decisions to onshore or offshore roles typically filled by H-1B visa workers.',
 'Tax may limit diversity by reducing H-1B visa hiring.',
 'Companies may need to reassess skill requirements due to H-1B visa tax.',
 'Tax may impact innovation driven by H-1B visa talent.',
 'Reduced H-1B visa hiring may limit research and development initiatives.',
 'Tax may disrupt the flow of innovative ideas from H-1B visa workers.',
 'Tax may hinder collaboration with international H-1B visa talent.',
 'Reduced H-1B visa workforce may slow technological advancements.']
 
# LLM Specification
llm_model = "openai::gpt-4o-mini"


# Specify Time Range
start_date = "2025-09-19"
end_date = "2025-10-06"

# Rerank Threshold
rerank_threshold = 0.7

# Search Frequency
freq = 'D'

# Fiscal Year
fiscal_year = 2025

# Document Limits
document_limit = 100

# Commen Params
common_params = {
    "narrative_sentences": main_narratives,
    "llm_model": llm_model,
    "start_date": start_date,
    "end_date": end_date,
    "rerank_threshold": rerank_threshold}

## Configure the Narrative Miners

Create narrative miners for each document type. In this example, we select MT Newswires as the news source.

In [10]:
# Common Params
common_params = {
    "narrative_sentences": main_narratives,
    "llm_model": llm_model,
    "start_date": start_date,
    "end_date": end_date,
    "rerank_threshold": rerank_threshold}
    

# Create the specialized miners for each document type
news_miner = NarrativeMiner(
    document_type=DocumentType.NEWS,
    fiscal_year=None,
    **common_params
)

## Run Narrative Mining

Execute the narrative mining processes for news:

In [11]:
# Mine news narratives
print("Mining news narratives...")
try:
    news_results = news_miner.mine_narratives(
        document_limit=document_limit,
        freq=freq,
        export_path=news_results_path
    )
    print("✅ News mining completed successfully!")
except Exception as e:
    print(f"Warning during news mining: {e}")


df = news_results["df_labeled"]
#df.to_csv("news_results_labeled_oct.csv", index=False)

Mining news narratives...


Querying OpenAI...: 100%|██████████| 12158/12158 [11:57<00:00, 16.94it/s]


✅ News mining completed successfully!


In [13]:
df = pd.read_csv("news_results_labeled_oct.csv")

df_narrative_search = df.copy()
#df_narrative_search = news_results["df_labeled"]


In [14]:
df_sentence_unique = df_narrative_search.drop_duplicates(subset=["Sentence ID"]).reset_index(drop=True)
print(len(df_narrative_search))
print(len(df_sentence_unique))

118587
5369


## Theme Match Labeling

This labeling is to filter out the retrieved chunks that are not about the main_theme

In [12]:
from src.labeler.narrative_labeler import NarrativeLabeler
labeler = NarrativeLabeler(llm_model=llm_model)


In [ ]:
df_sentence_unique_labeled = labeler.get_labels(
    main_theme="Introduction of a high fee on H-1B visas",
    texts=df_sentence_unique["Chunk Text"].tolist(),
    titles=df_sentence_unique["Headline"].tolist(),
    mode="theme_matching",
)

In [18]:
df_sentence_unique_labeled.rename(columns={"label": "Theme Matching Label", "motivation": "Theme Matching Motivation"}, inplace=True)
df_sentence_unique_labeled_merged = merge(df_sentence_unique, df_sentence_unique_labeled, left_index=True, right_index=True)

# Select columns from df_sentence_unique_labeled_filtered
columns_to_merge = df_sentence_unique_labeled_merged[['Sentence ID', 'Theme Matching Label', 'Theme Matching Motivation']]

# Merge with df on Sentence ID
df_sentences_merged = df.merge(columns_to_merge, on='Sentence ID', how='left')
df_sentences_related = df_sentences_merged[df_sentences_merged["Theme Matching Label"] != "unclear"]

df_sentences_related.to_csv("news_results_labeled.csv", index=False)



In [ ]:
#df_sentences_related = pd.read_csv("news_results_labeled.csv")

# Creating the DF

In [14]:
df = df_sentences_related.copy()
df_company = df[df["Entity Type"] == "COMP"]


In [15]:
# Donald Trump Dataframe
df_DonaldTrump = df[df["Entity"] == "Donald Trump"]

# US Dataframe
df_US = df[df["Country Code"] == "United States"]

In [16]:
#Top three People Entities after Donald Trump
counts = df[df["Entity Type"] == "PEOP"]["Entity"].value_counts()
top_3 = counts[counts.index != "Donald Trump"].head(3).index.tolist()

top_3_dict_entities = {}
for name in top_3:
    top_3_dict_entities[name] = df[df["Entity"] == name]

print(top_3_dict_entities.keys())

dict_keys(['Howard Lutnick', 'Jamie Dimon', 'Elon Musk'])


In [17]:
# Get top 3 companies from United States
counts = df_US[df_US["Entity Type"] == "COMP"]["Entity"].value_counts()
top_3_companies_US = counts.head(3).index.tolist()

# Create dictionary with dataframes for each top 5 company
top_3_companies_US_dict = {}
for company in top_3_companies_US:
    top_3_companies_US_dict[company] = df_US[df_US["Entity"] == company]

print(top_3_companies_US_dict.keys())

dict_keys(['Amazon.com Inc.', 'Microsoft Corp.', 'Alphabet Inc.'])


In [18]:
# Get value counts for places excluding India and United States
counts = df[df["Entity Type"] == "PLCE"]["Country Code"].value_counts()
top_3_places = counts[~counts.index.isin(["United States", "India"])].head(3).index.tolist()

# Create dictionary with dataframes for each top 3 place using Country Code
top_3_places_dict = {}
for place in top_3_places:
    top_3_places_dict[place] = df[df["Country Code"] == place]

print(top_3_places_dict.keys())

dict_keys(['China', 'Canada', 'United Kingdom'])


In [19]:
# Create dictionary of dictionaries with top 3 companies for each place
top_3_places_companies = {}

for place in top_3_places:
    df_place = top_3_places_dict[place]
    counts = df_place[df_place["Entity Type"] == "COMP"]["Entity"].value_counts()
    top_3_companies = counts.head(3).index.tolist()
    print(top_3_companies)
    companies_dict = {}
    for company in top_3_companies:
        companies_dict[company] = df_place[df_place["Entity"] == company]
    
    top_3_places_companies[place] = companies_dict

print(top_3_places_companies.keys())

['IBM Corp. Pty. Ltd.', 'Dewang', 'Metaview']
['Thomson Reuters Corp.', 'DBRS Ltd.', 'Talent Fund']
['The Financial Times Ltd.', 'Crossbridge Capital LLP', 'AJ Bell PLC']
dict_keys(['China', 'Canada', 'United Kingdom'])


# LABELING

In [20]:
from src.labeler.screener_labeler import ScreenerLabelerFlex
from src.labeler.narrative_labeler import NarrativeSummarizerFlex

## Trump Position

In [21]:
# Convert Date column to datetime for proper sorting
df_DonaldTrump_refined = df_DonaldTrump.copy()
df_DonaldTrump_refined['Date'] = pd.to_datetime(df_DonaldTrump_refined['Date'])

# Sort by Date (oldest first)
df_DonaldTrump_refined = df_DonaldTrump_refined.sort_values('Date')

# Remove duplicates keeping the first occurrence (oldest)
df_DonaldTrump_refined = df_DonaldTrump_refined.drop_duplicates(subset='Chunk Text', keep='first')

# Reset index
df_DonaldTrump_refined = df_DonaldTrump_refined.reset_index(drop=True)

print(len(df_DonaldTrump_refined))
print(len(df_DonaldTrump))


542
1781


In [22]:
import importlib
import sys

# Remove the module from cache if it exists
if 'src.labeler.narrative_labeler' in sys.modules:
    del sys.modules['src.labeler.narrative_labeler']

# Force reimport
from src.labeler.narrative_labeler import NarrativeLabeler
importlib.reload(sys.modules['src.labeler.narrative_labeler'])

import importlib
from src.prompts import labeler
importlib.reload(labeler)
from src.prompts.labeler import get_narrative_system_prompt
from src.prompts import labeler


## Extract Donald Trump direct references

In [28]:
labeler = NarrativeLabeler(llm_model=llm_model)
df_DT_citation = labeler.get_labels(
    main_theme="Introduction of a high fee on H-1B visas",
    texts=df_DonaldTrump_refined["Chunk Text"].tolist(),
    titles=df_DonaldTrump_refined["Headline"].tolist(),
    mode="entity_reference",
    entity_track="Donald Trump"
)


Querying OpenAI...:   0%|          | 0/542 [00:00<?, ?it/s]

Querying OpenAI...: 100%|██████████| 542/542 [00:22<00:00, 24.10it/s]


In [29]:
df_DT_theme_matching = merge(df_DonaldTrump_refined, df_DT_citation, left_index=True, right_index=True)
df_DT_theme_matching.loc[df_DT_theme_matching["label"] == "unclear", "label"] = "narrative"
df_DT_theme_matching_citation = df_DT_theme_matching[df_DT_theme_matching["label"] == "quote"]
df_DT_theme_matching_citation.replace(to_replace="unclear", value="narrative", inplace=True)

df_DT_theme_matching_narrative = df_DT_theme_matching[df_DT_theme_matching["label"] == "narrative"]

df_DT_theme_matching_citation.to_csv("news_results_labeled_citation.csv", index=False)
df_DT_theme_matching_narrative.to_csv("news_results_labeled_narrative.csv", index=False)

/tmp/ipykernel_26210/2741400982.py:4: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy



In [ ]:
#df_DT_theme_matching_citation = pd.read_csv("news_results_labeled_citation.csv")
#df_DT_theme_matching_narrative = pd.read_csv("news_results_labeled_narrative.csv")

In [24]:
import importlib
import sys

# Remove the module from cache if it exists
if 'src.labeler.narrative_labeler' in sys.modules:
    del sys.modules['src.labeler.narrative_labeler']

# Force reimport
from src.labeler.narrative_labeler import NarrativeLabeler, NarrativeSummarizerFlex
importlib.reload(sys.modules['src.labeler.narrative_labeler'])

import importlib
from src.prompts import labeler
importlib.reload(labeler)
from src.prompts.labeler import get_narrative_system_prompt
from src.prompts import labeler
from src.labeler.narrative_labeler import NarrativeLabeler, NarrativeSummarizerFlex


In [25]:
narrative_summarizer = NarrativeSummarizerFlex(llm_model="openai::gpt-4o")

# Sort and initialize columns once
df_DT_theme_matching_citation = df_DT_theme_matching_citation.sort_values('Date')
df_DT_theme_matching_citation["daily summary"] = None
df_DT_theme_matching_citation["daily bullet points"] = None

# Initialize cumulative dictionary
previous_narrative_dict = {}

narrative_dates = df_DT_theme_matching_citation["Date"].unique()

In [47]:


for date in narrative_dates:
    df_temp = df_DT_theme_matching_citation[df_DT_theme_matching_citation["Date"] == date]
    
    # Get daily recap
    daily_recap = narrative_summarizer.get_summaries(
        main_theme="Introduction of a high fee on H-1B visas",
        df=df_temp,
        mode="temporal_narrative", 
        entity_track="Donald Trump",
        previous_narrative=previous_narrative_dict
    )
    
    # Extract results
    summary = daily_recap["summary"].iloc[0]
    key_points = daily_recap["key_points"].iloc[0]
    
    # Convert key_points list to string with newlines
    if isinstance(key_points, list):
        key_points_str = "\n".join(key_points)
    else:
        key_points_str = str(key_points) if key_points is not None else ""
    
    print("Summary:", summary)
    print("Key points:", key_points_str)
    
    # Update dataframe - now using the string version
    mask = df_DT_theme_matching_citation["Date"] == date
    df_DT_theme_matching_citation.loc[mask, "daily summary"] = summary
    df_DT_theme_matching_citation.loc[mask, "daily bullet points"] = key_points_str
    
    # Add to cumulative dictionary for next iteration - CONVERT DATE TO STRING
    date_str = str(date)  # Convert Timestamp to string
    if isinstance(key_points, list):
        previous_narrative_dict[date_str] = summary + "\n" + "\n".join(key_points)
    else:
        previous_narrative_dict[date_str] = summary + "\n" + str(key_points)

df_DT_theme_matching_citation.to_csv("news_results_labeled_citation_summarized.csv", index=False)

DEBUG: SYSTEM PROMPT

Forget all previous prompts.
You are assisting a professional analyst in creating comprehensive summaries with bullet points based on some daily quotes about 'Donald Trump' coming from news articles.
The content of the summaries and the bullet points must only be related to the theme 'Introduction of a high fee on H-1B visas'. If some part of the information does not relate to 'Introduction of a high fee on H-1B visas', do not include it in the summary or in the bullet points.

You will receive:
    - Multiple sentences from news articles

Please adhere strictly to the following guidelines:

1. **Task**: 
    - Your task is to analyze today's quotes about 'Donald Trump'

2. **Create an updated summary that**:
    - Primarily focuses on the quotes from 'Donald Trump' 
    - Create a summary of the quotes from 'Donald Trump'

3. **Generate bullet points that**:
    - Contain the actual quotes (direct citations) from today's articles
    - Each bullet should include 

Querying OpenAI...: 100%|██████████| 1/1 [00:05<00:00,  5.04s/it]

Summary: Donald Trump expressed optimism about the tech industry's reaction to the introduction of a high fee on H-1B visas, which is aimed at ensuring that only highly skilled workers are brought into the U.S. The new fee of $100,000 is intended to discourage the replacement of American workers and promote the hiring of domestic talent.
Key points: "I think they're going to be very happy," Trump said, anticipating the tech industry's response to the changes.
"One of the most abused visa systems is the H1-B non-immigrant visa programme. This is supposed to allow highly skilled labourers who work in fields that Americans don't work in to come into the United States of America. What this proclamation will do is raise the fee that companies pay to sponsor H-1B applicants to $100,000."
"We need workers, we need great workers and this pretty much ensure that that's going to happen," Trump said.
DEBUG: SYSTEM PROMPT

Forget all previous prompts.
You are assisting a professional analyst in cr


Querying OpenAI...: 100%|██████████| 1/1 [00:07<00:00,  7.14s/it]

Summary: Donald Trump has reiterated his administration's commitment to imposing a high fee on H-1B visas, raising it from $215 to $100,000 annually. This significant increase is aimed at discouraging the hiring of foreign workers in favor of American talent. Trump believes that this move will ensure that companies prioritize hiring skilled American workers. He has claimed that the new fee will be beneficial for the U.S. workforce, despite concerns from tech companies about the potential negative impact on their operations. The administration has framed this fee increase as a response to what they describe as systemic abuse of the H-1B program.
Key points: President Donald Trump announced plans to impose a $100,000 annual fee on H-1B visas, threatening to upend the program that underpins America's technology workforce.
Trump argued that America needs 'great workers' and that the newly imposed fee 'pretty much ensures that that's what's going to happen.'
The move is expected to signific

...

Summary: Donald Trump has reiterated his administration's commitment to the $100,000 fee on H-1B visas, emphasizing its role in curbing program abuse and promoting the hiring of American workers. This fee is positioned as a one-time charge for new applications, which has caused confusion among tech companies. Trump believes that the fee will incentivize companies to prioritize American talent over foreign workers, despite concerns about its potential negative impact on industries reliant on skilled foreign labor. Analysts and company leaders have expressed concern about potential downsizing and workforce disruption, indicating that some companies might circumvent the fee by hiring talent in countries like India. This reflects a growing apprehension about the implications of the fee on the tech industry and the future of H-1B visa holders.
Key points: Trump claims companies will prefer to avoid paying the new fee, and hiring Americans would allow them to do so. "So there's an incentive 


Querying OpenAI...: 100%|██████████| 1/1 [00:06<00:00,  6.97s/it]

Summary: Donald Trump has announced a $100,000 fee for new H-1B visa applicants, reinforcing his administration's stance on curbing program abuse and promoting the hiring of American workers. This fee is intended to discourage the hiring of foreign workers and is framed as a necessary measure to protect U.S. jobs. The announcement has caused confusion among tech companies, which are concerned about the potential negative impact on their operations and may consider moving jobs overseas. Trump's assertion that the fee will encourage companies to prioritize American talent aligns with his previous statements regarding the need to address systemic abuses of the H-1B program.
Key points: On Friday, President Donald Trump's administration announced a $100,000 fee for all new recipients of H-1B visas, a type of nonimmigrant visa designed to help U.S. companies find employees with technical skills not common in the U.S.
Trump argued that the fee will encourage companies to hire American worker


Querying OpenAI...: 100%|██████████| 1/1 [00:07<00:00,  7.17s/it]

Summary: Donald Trump has reiterated his administration's commitment to the $100,000 fee on H-1B visas, emphasizing its role in curbing program abuse and promoting the hiring of American workers. This fee is intended to discourage the hiring of foreign workers and is framed as a necessary measure to protect U.S. jobs. The announcement has caused confusion among tech companies, which are concerned about the potential negative impact on their operations and may consider moving jobs overseas. Trump's assertion that the fee will encourage companies to prioritize American talent aligns with his previous statements regarding the need to address systemic abuses of the H-1B program. Notably, Ajay Jain Bhutoria, an Indian entrepreneur, has expressed skepticism about the fee's effectiveness, suggesting it may lead to increased outsourcing and hurt startups.
Key points: Donald Trump has reiterated his administration's commitment to the $100,000 fee on H-1B visas, emphasizing its role in curbing p

In [ ]:
#df_DT_theme_matching_citation = pd.read_csv("news_results_labeled_citation_summarized.csv")
#df_DT_theme_matching_citation.to_csv("news_results_labeled_citation_summarized.csv", index=False)

In [28]:
from src.report.html import generate_daily_recap_html
generate_daily_recap_html(df_DT_theme_matching_citation)

HTML file saved as: daily_evolution_recap.html


'daily_evolution_recap.html'

## US and India Company Position

In [29]:
narrative_dates

array(['2025-09-19', '2025-09-20', '2025-09-21', '2025-09-22',
       '2025-09-23', '2025-09-24', '2025-09-25'], dtype=object)

In [30]:
# Generalized: loop over a list of country codes, extract top 3 companies for each, and concatenate results

country_codes = ["United States", "India", "China","Canada" ]  # You can change this list as needed

# Convert the "Date" column to datetime (only once)
df["Date"] = pd.to_datetime(df["Date"])

# Dictionary: key = (country_code, company), value = dataframe of that company (over the whole date range)
top_3_companies_by_country = {}
top_3_companies_names_by_country = {}

for code in country_codes:
    # Filter by country code and narrative_dates
    df_country = df[
        (df["Country Code"] == code) &
        (df["Date"].isin(narrative_dates))
    ]
    # Get counts of all companies (Entity Type == "COMP") over the whole range
    company_counts = df_country[df_country["Entity Type"] == "COMP"]["Entity"].value_counts()
    top_3_companies = company_counts.head(3).index.tolist()
    top_3_companies_names_by_country[code] = top_3_companies

    # For each top company, save the dataframe filtered for that company (over all dates)
    for company in top_3_companies:
        top_3_companies_by_country[(code, company)] = df_country[df_country["Entity"] == company]

    # Print: the top 3 companies and their total occurrences for this country
    print(f"Top 3 companies for {code} and their total occurrences:")
    for company in top_3_companies:
        occ = len(top_3_companies_by_country[(code, company)])
        print(f"  {company}: {occ} occurrences")

# Unisci i dataframe dei top 3 per ogni country code in un unico dataframe
df_top_3_all_countries = pd.concat(
    [top_3_companies_by_country[(code, company)] 
     for code in country_codes 
     for company in top_3_companies_names_by_country[code]],
    ignore_index=True
)


Top 3 companies for United States and their total occurrences:
  Amazon.com Inc.: 493 occurrences
  Microsoft Corp.: 441 occurrences
  Alphabet Inc.: 327 occurrences
Top 3 companies for India and their total occurrences:
  Infosys Ltd.: 190 occurrences
  Wipro Ltd.: 133 occurrences
  Tata Consultancy Services Ltd.: 97 occurrences
Top 3 companies for China and their total occurrences:
  IBM Corp. Pty. Ltd.: 19 occurrences
  Dewang: 9 occurrences
  Metaview: 7 occurrences
Top 3 companies for Canada and their total occurrences:
  Thomson Reuters Corp.: 205 occurrences
  DBRS Ltd.: 8 occurrences
  Talent Fund: 2 occurrences


/tmp/ipykernel_150531/2811259160.py:16: FutureWarning:

The behavior of 'isin' with dtype=datetime64[ns] and castable values (e.g. strings) is deprecated. In a future version, these will not be considered matching by isin. Explicitly cast to the appropriate dtype before calling isin instead.

/tmp/ipykernel_150531/2811259160.py:16: FutureWarning:

The behavior of 'isin' with dtype=datetime64[ns] and castable values (e.g. strings) is deprecated. In a future version, these will not be considered matching by isin. Explicitly cast to the appropriate dtype before calling isin instead.

/tmp/ipykernel_150531/2811259160.py:16: FutureWarning:

The behavior of 'isin' with dtype=datetime64[ns] and castable values (e.g. strings) is deprecated. In a future version, these will not be considered matching by isin. Explicitly cast to the appropriate dtype before calling isin instead.

/tmp/ipykernel_150531/2811259160.py:16: FutureWarning:

The behavior of 'isin' with dtype=datetime64[ns] and castable 

In [36]:
narrative_summarizer = NarrativeSummarizerFlex(llm_model="openai::gpt-4o-mini")

# Sort and initialize columns once
df_top_3_all_countries = df_top_3_all_countries.sort_values('Date')
df_top_3_all_countries["summary"] = None
df_top_3_all_countries["key_points"] = None
df_top_3_all_countries["quotes"] = None

for date in narrative_dates:
    df_temp = df_top_3_all_countries[df_top_3_all_countries["Date"] == date]
    
    # Get daily recap
    daily_recap = narrative_summarizer.get_summaries(
        main_theme="Introduction of a high fee on H-1B visas",
        df=df_temp,
        mode="companies_impact",
        additional_parameters={"main_entity": "Donald Trump"}
    )
    
    # Merge dei risultati con il dataframe originale
    for _, row in daily_recap.iterrows():
        entity = row['Entity']
        
        # Trova gli indici delle righe che matchano
        mask = (df_top_3_all_countries["Date"] == date) & (df_top_3_all_countries["Entity"] == entity)
        indices = df_top_3_all_countries[mask].index
        
        # Assegna i valori a ogni riga individualmente
        for idx in indices:
            df_top_3_all_countries.at[idx, "summary"] = row['summary']
            df_top_3_all_countries.at[idx, "key_points"] = row['key_points']  
            df_top_3_all_countries.at[idx, "quotes"] = row['quotes']  

DEBUG: SYSTEM PROMPT

Forget all previous prompts.
You are assisting a professional analyst in creating comprehensive company summaries, bullet points and direct quotes about the 'Introduction of a high fee on H-1B visas' introduced by 'Donald Trump'.
Your task is to analyze multiple piece of sentences coming from news for a single company about a single day.

You will receive company data including:
- Company name
- Multiple sentences from news articles

Your primary task is to synthesize this information into a comprehensive summary on how the company is positioned regarding the 'Introduction of a high fee on H-1B visas' and to extract only direct quotes coming from people of the given company.
Put a lot of attention and highlight any reposne or action or comment from the given company about the 'Introduction of a high fee on H-1B visas'.
If some part of the information does not relate to 'Introduction of a high fee on H-1B visas' or does not come from the given company, do not inclu

Querying OpenAI...: 100%|██████████| 6/6 [00:07<00:00,  1.20s/it]

DEBUG: SYSTEM PROMPT

Forget all previous prompts.
You are assisting a professional analyst in creating comprehensive company summaries, bullet points and direct quotes about the 'Introduction of a high fee on H-1B visas' introduced by 'Donald Trump'.
Your task is to analyze multiple piece of sentences coming from news for a single company about a single day.

You will receive company data including:
- Company name
- Multiple sentences from news articles

Your primary task is to synthesize this information into a comprehensive summary on how the company is positioned regarding the 'Introduction of a high fee on H-1B visas' and to extract only direct quotes coming from people of the given company.
Put a lot of attention and highlight any reposne or action or comment from the given company about the 'Introduction of a high fee on H-1B visas'.
If some part of the information does not relate to 'Introduction of a high fee on H-1B visas' or does not come from the given company, do not inclu


Querying OpenAI...:  12%|█▎        | 1/8 [00:04<00:29,  4.14s/it]


KeyboardInterrupt: 

In [34]:
# Crea dataframe con le informazioni estratte
df_company_summaries = df_top_3_all_countries[df_top_3_all_countries['summary'].notna()][['Entity', 'Date', 'summary', 'key_points', 'quotes',"Country Code"]].drop_duplicates(subset=['Entity', 'Date']).sort_values(['Entity', 'Date']).reset_index(drop=True)

df_company_summaries.columns = ['Entity', 'Date', 'Summary', 'Key_points', 'Quotes','Country']

df_company_summaries.to_csv("news_results_labeled_company_summaries.csv", index=False)

df_company_summaries

,Entity,Date,Summary,Key_points,Quotes,Country
0,Alphabet Inc.,2025-09-19,Alphabet Inc. is positioned to face significan...,[The introduction of a high fee on H-1B visas ...,"[For companies like Google, Microsoft, Amazon,...",United States
1,Alphabet Inc.,2025-09-20,Alphabet Inc. is positioned to face significan...,"[The $100,000 annual fee on H-1B visas is expe...","[""US tech giants like Amazon, Microsoft, Googl...",United States
2,Alphabet Inc.,2025-09-21,Alphabet Inc. is significantly impacted by the...,[Alphabet Inc. is concerned about the financia...,"[Amazon, Alphabet's Google, Microsoft and othe...",United States
3,Alphabet Inc.,2025-09-22,"Alphabet Inc., through its subsidiary Google, ...",[Alphabet Inc.'s Google employs thousands of H...,"[In response to the announcement, tech giants ...",United States
4,Alphabet Inc.,2025-09-23,Alphabet Inc. is positioned as a significant p...,[Alphabet Inc. is among the tech giants that h...,"[For Silicon Valley, the decision is disruptiv...",United States
5,Alphabet Inc.,2025-09-24,Alphabet Inc. is significantly impacted by the...,[Alphabet Inc. and other major U.S. tech compa...,[The sharp increase in fees will double hiring...,United States
6,Alphabet Inc.,2025-09-25,Alphabet Inc. is positioned as a major player ...,"[The $100,000 application fee for H-1B visas w...","[Major companies like Microsoft, Google, and A...",United States
7,Amazon.com Inc.,2025-09-19,Amazon.com Inc. is significantly impacted by t...,"[Amazon, along with other big tech companies, ...",[The proposal has been met with mixed reviews....,United States
8,Amazon.com Inc.,2025-09-20,Amazon.com Inc. is significantly impacted by t...,[Amazon has warned employees on H-1B and H-4 v...,[Amazon has warned employees on H-1B and H-4 v...,United States
9,Amazon.com Inc.,2025-09-21,Amazon.com Inc. is significantly impacted by t...,[Amazon has tens of thousands of staff in the ...,"[Following the announcement, companies such as...",United States


In [35]:
narrative_summarizer = NarrativeSummarizerFlex(llm_model="openai::gpt-4o-mini")

# Sort dataframe
df_company_summaries = df_company_summaries.sort_values(['Entity', 'Date'])
df_company_summaries["enhanced_summary"] = None
df_company_summaries["enhanced_key_points"] = None

# Initialize global cumulative dictionary with "entity_date" keys
previous_narrative_dict = {}

# Get unique entities and process each entity across all its dates
for entity in df_company_summaries["Entity"].unique():
    entity_df = df_company_summaries[df_company_summaries["Entity"] == entity]
    dates = sorted(entity_df["Date"].unique())
    
    # Variables to track summaries for consolidation
    iteration_summaries = {}  # Store summary for each iteration
    consolidated_narrative = None
    
    # Process each date for this entity
    for iteration, date in enumerate(dates):
        df_temp = entity_df[entity_df["Date"] == date]
        
        print(f"Processing Entity: {entity}, Date: {date}, Iteration: {iteration}")
        
        if iteration == 0:
            # First iteration: skip, use only existing summary
            print("First iteration - skipping get_summaries, using existing summary")
            
            # Take existing summary as baseline
            existing_summary = df_temp["Summary"].iloc[0] if "Summary" in df_temp.columns else ""
            existing_key_points = df_temp["Key_points"].iloc[0] if "Key_points" in df_temp.columns else []
            
            # Ensure key_points is a list
            if not isinstance(existing_key_points, list):
                existing_key_points = [str(existing_key_points)] if existing_key_points else []
            
            # Save to dataframe (use .at to assign list to single cell)
            mask = (df_company_summaries["Date"] == date) & (df_company_summaries["Entity"] == entity)
            row_index = df_company_summaries[mask].index[0]
            df_company_summaries.at[row_index, "enhanced_summary"] = existing_summary
            df_company_summaries.at[row_index, "enhanced_key_points"] = existing_key_points
            
            # Store iteration 0 summary (convert to string for consolidation purposes)
            key_points_str = "\n".join(existing_key_points) if existing_key_points else ""
            iteration_summaries[0] = existing_summary + "\n" + key_points_str
            consolidated_narrative = iteration_summaries[0]
            
        elif iteration == 1:
            # Second iteration: only temporal analysis with previous narrative
            print("Second iteration - temporal analysis only")
            
            # Prepare previous narrative for temporal analysis
            entity_previous_narrative = {f"{entity}_{dates[0]}": consolidated_narrative}
            print("entity_previous_narrative:",entity_previous_narrative)

            daily_recap = narrative_summarizer.get_summaries(
                main_theme="Introduction of a high fee on H-1B visas",
                df=df_temp,
                mode="temporal_company_narrative_from_summaries",  
                previous_narrative=entity_previous_narrative
            )
            
            # Process results
            for _, recap_row in daily_recap.iterrows():
                entity_name = recap_row['Entity']
                summary = recap_row['summary']
                key_points = recap_row['key_points']
                
                # Ensure key_points is a list
                if not isinstance(key_points, list):
                    key_points = [str(key_points)] if key_points else []
                
                # Update dataframe (use .at to assign list to single cell)
                mask = (df_company_summaries["Date"] == date) & (df_company_summaries["Entity"] == entity_name)
                row_index = df_company_summaries[mask].index[0]
                df_company_summaries.at[row_index, "enhanced_summary"] = summary
                df_company_summaries.at[row_index, "enhanced_key_points"] = key_points
                
                # Store iteration 1 summary (convert to string for consolidation purposes)
                key_points_str = "\n".join(key_points) if key_points else ""
                iteration_summaries[1] = summary + "\n" + key_points_str
                
        else:
            # Third iteration and beyond: consolidate then do temporal analysis
            print(f"Iteration {iteration} - consolidation + temporal analysis")
            
            # Step 1: Consolidate previous summaries
            # For iteration 2: previous = iteration 0, today = iteration 1
            # For iteration 3+: previous = consolidated from previous step, today = previous iteration
            if iteration == 2:
                previous_summary_for_consolidation = iteration_summaries[0]
                today_summary_for_consolidation = iteration_summaries[1]
            else:
                previous_summary_for_consolidation = consolidated_narrative
                today_summary_for_consolidation = iteration_summaries[iteration - 1]
            
            print(f"Consolidating: previous from iteration {0 if iteration == 2 else 'consolidated'}, today from iteration {iteration - 1}")
            
            # Apply consolidation using new interface
            consolidated_narrative = narrative_summarizer.get_summaries(
                main_theme="Introduction of a high fee on H-1B visas",
                mode="company_narrative_consolidation",
                previous_summary=previous_summary_for_consolidation,
                today_summary=today_summary_for_consolidation
            )
            
            print(f"Consolidated narrative length: {len(consolidated_narrative) if consolidated_narrative else 0}")
            
            # Step 2: Apply temporal analysis with consolidated summary
            entity_previous_narrative = {f"{entity}_consolidated": consolidated_narrative}
            
            daily_recap = narrative_summarizer.get_summaries(
                main_theme="Introduction of a high fee on H-1B visas",
                df=df_temp,
                mode="temporal_company_narrative_from_summaries",  
                previous_narrative=entity_previous_narrative
            )
            
            # Process results
            for _, recap_row in daily_recap.iterrows():
                entity_name = recap_row['Entity']
                summary = recap_row['summary']
                key_points = recap_row['key_points']
                
                # Ensure key_points is a list
                if not isinstance(key_points, list):
                    key_points = [str(key_points)] if key_points else []
                
                # Update dataframe (use .at to assign list to single cell)
                mask = (df_company_summaries["Date"] == date) & (df_company_summaries["Entity"] == entity_name)
                row_index = df_company_summaries[mask].index[0]
                df_company_summaries.at[row_index, "enhanced_summary"] = summary
                df_company_summaries.at[row_index, "enhanced_key_points"] = key_points
                
                # Store current iteration summary (convert to string for consolidation purposes)
                key_points_str = "\n".join(key_points) if key_points else ""
                iteration_summaries[iteration] = summary + "\n" + key_points_str
        
        print(f"Completed Entity: {entity}, Date: {date}")
        print("-" * 50)

Processing Entity: Alphabet Inc., Date: 2025-09-19 00:00:00, Iteration: 0
First iteration - skipping get_summaries, using existing summary
Completed Entity: Alphabet Inc., Date: 2025-09-19 00:00:00
--------------------------------------------------
Processing Entity: Alphabet Inc., Date: 2025-09-20 00:00:00, Iteration: 1
Second iteration - temporal analysis only
entity_previous_narrative: {'Alphabet Inc._2025-09-19 00:00:00': 'Alphabet Inc. is positioned to face significant challenges due to the introduction of a high fee on H-1B visas. The company, along with other major tech firms like Google, Microsoft, and Amazon, may need to rethink their hiring practices as the new fee could make it more difficult for foreign workers to access U.S. job opportunities. This change is expected to impact not only new visa applications but also renewals, potentially limiting the talent pool for these companies.\nThe introduction of a high fee on H-1B visas could lead to a dramatic rethink of hiring pr

Querying OpenAI...:   0%|          | 0/1 [00:00<?, ?it/s]

Querying OpenAI...: 100%|██████████| 1/1 [00:08<00:00,  8.04s/it]


Completed Entity: Alphabet Inc., Date: 2025-09-20 00:00:00
--------------------------------------------------
Processing Entity: Alphabet Inc., Date: 2025-09-21 00:00:00, Iteration: 2
Iteration 2 - consolidation + temporal analysis
Consolidating: previous from iteration 0, today from iteration 1
DEBUG: CONSOLIDATION SYSTEM PROMPT

Forget all previous prompts.
You are assisting a professional analyst in extracting and creating information from a series of info about the theme of 'Introduction of a high fee on H-1B visas' coming from news articles.
Your are given summarized infomation about the theme of 'Introduction of a high fee on H-1B visas' coming from news articles about past days.
You are also given a summarized version of the information about the theme of 'Introduction of a high fee on H-1B visas' coming from news articles about today.

1. **Task**: 
    - You must extract the new information from the new information and add it to the summary of the information about the theme o

Querying OpenAI...: 100%|██████████| 1/1 [00:04<00:00,  4.73s/it]

...

Completed Entity: Wipro Ltd., Date: 2025-09-24 00:00:00
--------------------------------------------------
Processing Entity: Wipro Ltd., Date: 2025-09-25 00:00:00, Iteration: 6
Iteration 6 - consolidation + temporal analysis
Consolidating: previous from iteration consolidated, today from iteration 5
DEBUG: CONSOLIDATION SYSTEM PROMPT

Forget all previous prompts.
You are assisting a professional analyst in extracting and creating information from a series of info about the theme of 'Introduction of a high fee on H-1B visas' coming from news articles.
Your are given summarized infomation about the theme of 'Introduction of a high fee on H-1B visas' coming from news articles about past days.
You are also given a summarized version of the information about the theme of 'Introduction of a high fee on H-1B visas' coming from news articles about today.

1. **Task**: 
    - You must extract the new information from the new information and add it to the summary of the information about the th

Querying OpenAI...: 100%|██████████| 1/1 [00:17<00:00, 17.56s/it]

Consolidated narrative length: 5876
DEBUG: SYSTEM PROMPT

Forget all previous prompts.
You are assisting a professional analyst in creating comprehensive summaries based on some daily summaries, bullet points and quotes about the theme of 'Introduction of a high fee on H-1B visas' created from news articles.
You are also given a cumulative narrative summary from recent past days about 'Introduction of a high fee on H-1B visas'. That summary was generated by previous iterations of this same prompt and represent the historical context that you can use to analyze and recognize the new additional information.
Your goal is to create a new summary only on the new information that has not been mentioned in the previous summaries, in order to keep track of the new information that has not been mentioned in the previous summaries.


You will receive:
    - Historical narrative summary from recent past days about 'Introduction of a high fee on H-1B visas'
    - Summaries, bullet points and quote


Querying OpenAI...: 100%|██████████| 1/1 [00:05<00:00,  5.30s/it]

Completed Entity: Wipro Ltd., Date: 2025-09-25 00:00:00
--------------------------------------------------


In [38]:
df_company_summaries.to_csv("news_results_labeled_company_summaries.csv", index=False)

In [32]:
df_company_summaries

,Entity,Date,Summary,Key_points,Quotes,Country,enhanced_summary,enhanced_key_points
0,Alphabet Inc.,2025-09-19,Alphabet Inc. is positioned to face significan...,['The introduction of a high fee on H-1B visas...,"['For companies like Google, Microsoft, Amazon...",United States,Alphabet Inc. is positioned to face significan...,['The introduction of a high fee on H-1B visas...
1,Alphabet Inc.,2025-09-20,Alphabet Inc. is positioned to face significan...,"['The $100,000 annual fee on H-1B visas is exp...","['""US tech giants like Amazon, Microsoft, Goog...",United States,Alphabet Inc. is now facing the introduction o...,"['The introduction of a $100,000 annual fee on..."
2,Alphabet Inc.,2025-09-21,Alphabet Inc. is significantly impacted by the...,['Alphabet Inc. is concerned about the financi...,"[""Amazon, Alphabet's Google, Microsoft and oth...",United States,Alphabet Inc. is facing significant financial ...,['Alphabet Inc. is concerned about the financi...
3,Alphabet Inc.,2025-09-22,"Alphabet Inc., through its subsidiary Google, ...","[""Alphabet Inc.'s Google employs thousands of ...","['In response to the announcement, tech giants...",United States,"Alphabet Inc., through its subsidiary Google, ...",['Alphabet Inc. employs thousands of H-1B visa...
4,Alphabet Inc.,2025-09-23,Alphabet Inc. is positioned as a significant p...,['Alphabet Inc. is among the tech giants that ...,"['For Silicon Valley, the decision is disrupti...",United States,New information indicates that Alphabet Inc. h...,['Alphabet Inc. has historically relied on H-1...
5,Alphabet Inc.,2025-09-24,Alphabet Inc. is significantly impacted by the...,['Alphabet Inc. and other major U.S. tech comp...,['The sharp increase in fees will double hirin...,United States,New information indicates that the introductio...,['The new fee on H-1B visas is expected to dou...
6,Alphabet Inc.,2025-09-25,Alphabet Inc. is positioned as a major player ...,"['The $100,000 application fee for H-1B visas ...","['Major companies like Microsoft, Google, and ...",United States,Alphabet Inc. continues to face significant ch...,"['The introduction of the $100,000 fee on H-1B..."
7,Amazon.com Inc.,2025-09-19,Amazon.com Inc. is significantly impacted by t...,"['Amazon, along with other big tech companies,...",['The proposal has been met with mixed reviews...,United States,Amazon.com Inc. is significantly impacted by t...,"['Amazon, along with other big tech companies,..."
8,Amazon.com Inc.,2025-09-20,Amazon.com Inc. is significantly impacted by t...,['Amazon has warned employees on H-1B and H-4 ...,['Amazon has warned employees on H-1B and H-4 ...,United States,Amazon.com Inc. is facing significant challeng...,['Amazon has warned employees on H-1B and H-4 ...
9,Amazon.com Inc.,2025-09-21,Amazon.com Inc. is significantly impacted by t...,['Amazon has tens of thousands of staff in the...,"['Following the announcement, companies such a...",United States,Amazon.com Inc. continues to face significant ...,['Amazon has tens of thousands of staff in the...


In [33]:
from src.report.html import generate_company_comparison_html

generate_company_comparison_html(df_company_summaries)

Company comparison HTML file saved as: company_comparison_report.html


'company_comparison_report.html'

## Generate Final Narrative for each of the Actors

In [34]:
df_temp_summaries = df_company_summaries.copy()

In [35]:
df_summaries_companies = df_company_summaries[["Entity", "Date", "enhanced_summary", "enhanced_key_points"]]
df_summaries_companies = df_summaries_companies.rename(columns={"enhanced_summary": "Summary", "enhanced_key_points": "Key Points"})
df_summaries_trump = df_DT_theme_matching_citation[["Entity", "Date", "daily summary", "daily bullet points"]]
df_summaries_trump = df_summaries_trump.drop_duplicates(subset=["Date"])
df_summaries_trump = df_summaries_trump.rename(columns={"daily summary": "Summary", "daily bullet points": "Key Points"})
df_summaries_entities = pd.concat([df_summaries_companies, df_summaries_trump], ignore_index=True)

In [ ]:
#df_summaries_entities.to_csv("df_summaries_entities.csv", index=False)

In [93]:
df_summaries_entities["Key Points"].iloc[63]

"On Friday, President Donald Trump's administration announced a $100,000 fee for all new recipients of H-1B visas, a type of nonimmigrant visa designed to help U.S. companies find employees with technical skills not common in the U.S.\nTrump argued that the fee will encourage companies to hire American workers, instead of foreign ones.\nThe Trump administration's sharp increase in visa fees for H-1B workers has triggered concern among US technology companies, with many considering moving more jobs overseas.\nTrump stated that 'abuse' of H-1B visas has facilitated an 'influx' of foreign labour in STEM that has undermined salaries and workplace conditions for US workers.\nThe fee hike sent H-1B holders into panic mode, prompting companies like Microsoft to ask employees abroad to rush back before midnight."

In [39]:
df_summaries_entities['Date'] = pd.to_datetime(df_summaries_entities['Date']).dt.strftime('%Y-%m-%d')


ValueError: time data "2025-09-19" doesn't match format "%Y-%m-%d %H:%M:%S", at position 58. You might want to try:
    - passing `format` if your strings have a consistent format;
    - passing `format='ISO8601'` if your strings are all ISO8601 but not necessarily in exactly the same format;
    - passing `format='mixed'`, and the format will be inferred for each element individually. You might want to use `dayfirst` alongside this.

In [53]:
import importlib
import sys

# Remove the module from cache if it exists
if 'src.labeler.narrative_labeler' in sys.modules:
    del sys.modules['src.labeler.narrative_labeler']

# Force reimport
from src.labeler.narrative_labeler import NarrativeLabeler, NarrativeSummarizerFlex
importlib.reload(sys.modules['src.labeler.narrative_labeler'])

import importlib
from src.prompts import labeler
importlib.reload(labeler)
from src.prompts.labeler import get_narrative_system_prompt
from src.prompts import labeler
from src.labeler.narrative_labeler import NarrativeLabeler, NarrativeSummarizerFlex

In [63]:
narrative_summarizer = NarrativeSummarizerFlex(llm_model="openai::gpt-4o")

# Sort and initialize columns once
df_summaries_entities = df_summaries_entities.sort_values('Date')

    
# Get final recap
entities_final_summary = narrative_summarizer.get_summaries(
    main_theme="Introduction of a high fee on H-1B visas",
    df=df_summaries_entities,
    mode="final_summary_from_daily_summaries",
)



DEBUG: SYSTEM PROMPT

Forget all previous prompts.
You are assisting a professional analyst in creating a comprehensive final recap summary from daily summaries focused on the theme of 'Introduction of a high fee on H-1B visas'. Each daily summary is compiled from news articles that reference specific companies or individuals based on their strategic positioning, actions taken, or relationship to the main theme.
You have to create a final narrative summary that captures the most significant developments and insights from the daily summaries.

You are given:
    - Name of the company or individual
    - A series of dates and daily summaries

1. **Task**: 
    - You must create a final narrative summary that captures the most significant developments and insights from the daily summaries.
    - Do not include any information that is not related to the theme of 'Introduction of a high fee on H-1B visas' coming from news articles.
    - Do not add any information from your own knowledge or

Querying OpenAI...:   0%|          | 0/13 [00:00<?, ?it/s]

Querying OpenAI...: 100%|██████████| 13/13 [00:07<00:00,  1.71it/s]


In [65]:
entities_final_summary.to_csv("final_summary_per_entity.csv", index=False)

In [135]:
entities_final_summary = pd.read_csv("final_summary_per_entity.csv")

In [41]:
# Versione più elegante con pandas groupby
countries_dict = df_company_summaries.groupby('Country')['Entity'].apply(list).to_dict()

# Rimuovi eventuali duplicati (se ci sono)
for country in countries_dict:
    countries_dict[country] = list(set(countries_dict[country]))


In [69]:
df_highlights = pd.DataFrame()
for country, companies in countries_dict.items():
    print("Processing companies:", companies)
    daily_highlights = narrative_summarizer.get_summaries(
        main_theme="Introduction of a high fee on H-1B visas",
        df=df_summaries_entities,
        mode="companies_daily_highlights_from_daily_key_points",
        additional_parameters={"companies": companies}
    )
    df_highlights = pd.concat([df_highlights, daily_highlights], ignore_index=True)

Processing companies: ['Thomson Reuters Corp.', 'DBRS Ltd.', 'Talent Fund']


Querying OpenAI...:   0%|          | 0/1 [00:00<?, ?it/s]

Querying OpenAI...: 100%|██████████| 1/1 [00:03<00:00,  3.22s/it]


Processing companies: ['IBM Corp. Pty. Ltd.', 'Metaview', 'Dewang']


Querying OpenAI...: 100%|██████████| 1/1 [00:02<00:00,  2.89s/it]


Processing companies: ['Tata Consultancy Services Ltd.', 'Infosys Ltd.', 'Wipro Ltd.']


Querying OpenAI...: 100%|██████████| 1/1 [00:09<00:00,  9.26s/it]


Processing companies: ['Amazon.com Inc.', 'Alphabet Inc.', 'Microsoft Corp.']


Querying OpenAI...: 100%|██████████| 1/1 [00:05<00:00,  5.19s/it]


In [70]:
df_highlights.to_csv("df_highlights.csv", index=False)

In [42]:
df_highlights = pd.read_csv("df_highlights.csv")

# CREATE VISUALIZATIONS

In [43]:
narrative_dates

array(['2025-09-19', '2025-09-20', '2025-09-21', '2025-09-22',
       '2025-09-23', '2025-09-24', '2025-09-25'], dtype=object)

In [147]:
df_company_summaries.head()

,Entity,Date,Summary,Key_points,Quotes,Country,enhanced_summary,enhanced_key_points
0,Alphabet Inc.,2025-09-19,Alphabet Inc. is positioned to face significan...,['The introduction of a high fee on H-1B visas...,"['For companies like Google, Microsoft, Amazon...",United States,Alphabet Inc. is positioned to face significan...,['The introduction of a high fee on H-1B visas...
1,Alphabet Inc.,2025-09-20,Alphabet Inc. is positioned to face significan...,"['The $100,000 annual fee on H-1B visas is exp...","['""US tech giants like Amazon, Microsoft, Goog...",United States,Alphabet Inc. is now facing the introduction o...,"['The introduction of a $100,000 annual fee on..."
2,Alphabet Inc.,2025-09-21,Alphabet Inc. is significantly impacted by the...,['Alphabet Inc. is concerned about the financi...,"[""Amazon, Alphabet's Google, Microsoft and oth...",United States,Alphabet Inc. is facing significant financial ...,['Alphabet Inc. is concerned about the financi...
3,Alphabet Inc.,2025-09-22,"Alphabet Inc., through its subsidiary Google, ...","[""Alphabet Inc.'s Google employs thousands of ...","['In response to the announcement, tech giants...",United States,"Alphabet Inc., through its subsidiary Google, ...",['Alphabet Inc. employs thousands of H-1B visa...
4,Alphabet Inc.,2025-09-23,Alphabet Inc. is positioned as a significant p...,['Alphabet Inc. is among the tech giants that ...,"['For Silicon Valley, the decision is disrupti...",United States,New information indicates that Alphabet Inc. h...,['Alphabet Inc. has historically relied on H-1...


In [149]:
import importlib
import sys

# Force reload of the plots module
if 'src.report.plots' in sys.modules:
    importlib.reload(sys.modules['src.report.plots'])

# Import the function
from src.report.plots import create_enhanced_interactive_chart

# Create enhanced chart
fig_enhanced = create_enhanced_interactive_chart(
    df_DT_theme_matching_narrative, 
    df_DT_theme_matching_citation, 
    df_top_3_all_countries, 
    df_company_summaries, 
    narrative_dates
)

# Show the enhanced plot
fig_enhanced.show()

# Save with better name
fig_enhanced.write_html("h1b_narrative_dashboard_enhanced.html")
print("Enhanced interactive dashboard saved as 'h1b_narrative_dashboard_enhanced.html'")

Enhanced interactive dashboard saved as 'h1b_narrative_dashboard_enhanced.html'


In [94]:

# Forza reload
import importlib
import sys
if 'src.report.html' in sys.modules:
    importlib.reload(sys.modules['src.report.html'])
from src.report import html

html.generate_interactive_timeline_dashboard_html(
    df_highlights=df_highlights,
    countries_dict=countries_dict,
    df_company_summaries=df_company_summaries,
    plotly_fig=fig_enhanced,
    output_file="simple_interactive_timeline.html"
    # Nessun parametro follow_trend!
)

Interactive timeline dashboard generated: simple_interactive_timeline.html


'simple_interactive_timeline.html'

Interactive timeline dashboard generated: full_dashboard.html


'full_dashboard.html'

In [55]:
countries_dict

{'Canada': ['Thomson Reuters Corp.', 'DBRS Ltd.', 'Talent Fund'],
 'China': ['Metaview', 'IBM Corp. Pty. Ltd.', 'Dewang'],
 'India': ['Tata Consultancy Services Ltd.', 'Infosys Ltd.', 'Wipro Ltd.'],
 'United States': ['Microsoft Corp.', 'Amazon.com Inc.', 'Alphabet Inc.']}

In [50]:
df_sentences_related.columns

Index(['Time Period', 'Date', 'Document ID', 'Sentence ID', 'Headline',
       'Chunk Text', 'Motivation', 'Label', 'Entity', 'Country Code',
       'Entity Type', 'Theme Matching Label', 'Theme Matching Motivation'],
      dtype='object')

In [52]:
unique_documents_count = df_sentences_related['Document ID'].nunique()
unique_documents_count

1372

In [76]:
unique_sentences_count = df_sentences_related['Sentence ID'].nunique()
unique_sentences_count

2175

In [74]:
# Extract all companies from countries_dict
all_companies = []
for country, companies in countries_dict.items():
    all_companies.extend(companies)

# Calculate overall totals for each company first
overall_totals = {}
for company in all_companies:
    company_rows = df_sentences_related[df_sentences_related['Entity'] == company]
    overall_totals[company] = {
        'Overall_Total_Sentences': len(company_rows),
        'Overall_Unique_Documents': company_rows['Document ID'].nunique()
    }

# Get unique dates
unique_dates = df_sentences_related['Date'].unique()

# Calculate statistics for each company and date
company_data = []
for date in unique_dates:
    # Filter data for this specific date
    date_data = df_sentences_related[df_sentences_related['Date'] == date]
    total_unique_docs_date = date_data['Document ID'].nunique()
    
    for company in all_companies:
        company_rows = date_data[date_data['Entity'] == company]
        total_sentences = len(company_rows)
        unique_docs = company_rows['Document ID'].nunique()
        
        # Calculate overall percentage
        overall_percentage = round((overall_totals[company]['Overall_Unique_Documents'] / unique_documents_count * 100) if unique_documents_count > 0 else 0, 2)
        
        company_data.append({
            'Date': date,
            'Company': company,
            'Total_Sentences': total_sentences,
            'Unique_Documents': unique_docs,
            'Total_Unique_Documents': total_unique_docs_date,
            'Percentage_Documents': round((unique_docs / total_unique_docs_date * 100) if total_unique_docs_date > 0 else 0, 2),
            'Overall_Total_Sentences': overall_totals[company]['Overall_Total_Sentences'],
            'Overall_Unique_Documents': overall_totals[company]['Overall_Unique_Documents'],
            'Overall_Percentage_Documents': overall_percentage
        })

# Create final dataframe
company_stats = pd.DataFrame(company_data)
company_stats = company_stats.sort_values(['Date', 'Total_Sentences'], ascending=[True, False])

In [91]:
company_stats

,Date,Company,Total_Sentences,Unique_Documents,Total_Unique_Documents,Percentage_Documents,Overall_Total_Sentences,Overall_Unique_Documents,Overall_Percentage_Documents
9,2025-09-19,Microsoft Corp.,9,4,29,13.79,441,165,12.03
10,2025-09-19,Amazon.com Inc.,9,4,29,13.79,493,169,12.32
11,2025-09-19,Alphabet Inc.,7,2,29,6.90,327,98,7.14
4,2025-09-19,IBM Corp. Pty. Ltd.,4,1,29,3.45,19,10,0.73
7,2025-09-19,Infosys Ltd.,4,1,29,3.45,190,79,5.76
...,...,...,...,...,...,...,...,...,...
74,2025-09-25,Talent Fund,1,1,63,1.59,2,2,0.15
75,2025-09-25,Metaview,0,0,63,0.00,7,1,0.07
76,2025-09-25,IBM Corp. Pty. Ltd.,0,0,63,0.00,19,10,0.73
77,2025-09-25,Dewang,0,0,63,0.00,9,2,0.15


In [70]:
# Force reload if needed
import importlib
import sys
if 'src.report.plots' in sys.modules:
    importlib.reload(sys.modules['src.report.plots'])

# Import the new function
from src.report.plots import create_company_document_coverage_chart

# Use it
fig_document_coverage = create_company_document_coverage_chart(company_stats)
fig_document_coverage.show()

In [71]:
fig2 = create_company_document_coverage_chart(company_stats, relative_overall=True)
fig2.show()

In [ ]:
df_summaries_entities.columns
df_summaries_trump.columns

Index(['Entity', 'Date', 'Summary', 'Key Points'], dtype='object')

In [112]:
df_company_summaries.columns

Index(['Entity', 'Date', 'Summary', 'Key_points', 'Quotes', 'Country',
       'enhanced_summary', 'enhanced_key_points'],
      dtype='object')

In [100]:
import importlib
import sys
if 'src.report.html' in sys.modules:
    importlib.reload(sys.modules['src.report.html'])
from src.report import html
from src.report.html import generate_interactive_timeline_dashboard_html

generate_interactive_timeline_dashboard_html(
    df_highlights=df_highlights,
    df_company_summaries=df_company_summaries,
    plotly_fig=fig_enhanced,
    countries_dict=countries_dict,
    df_entities_final_summary=entities_final_summary,
    company_stats=company_stats,  # ← NEW!
    unique_sentences_count=unique_sentences_count,  # ← NEW!
    output_file="full_dashboard.html"
)


Interactive timeline dashboard generated: full_dashboard.html


'full_dashboard.html'

In [ ]:
df_summaries_entities

In [150]:
# Force reload
import importlib
import sys
if 'src.report.html' in sys.modules:
    importlib.reload(sys.modules['src.report.html'])

from src.report.html import generate_interactive_timeline_dashboard_html

# Convert Date column handling mixed formats
df_summaries_entities["Date"] = pd.to_datetime(df_summaries_entities["Date"], format='mixed')

# Generate dashboard with People Mode using df_summaries_entities
generate_interactive_timeline_dashboard_html(
    df_highlights=df_highlights,
    df_company_summaries=df_company_summaries,
    plotly_fig=fig_enhanced,
    countries_dict=countries_dict,
    df_entities_final_summary=entities_final_summary,
    company_stats=company_stats,
    unique_sentences_count=unique_sentences_count,
    df_summaries_entities=df_summaries_entities,
    output_file="full_dashboard_with_people.html"
)

Interactive timeline dashboard generated: full_dashboard_with_people.html


'full_dashboard_with_people.html'

# Generate unique final Summary 

In [127]:
narrative_dates

array(['2025-09-19', '2025-09-20', '2025-09-21', '2025-09-22',
       '2025-09-23', '2025-09-24', '2025-09-25'], dtype=object)

In [136]:
import importlib
import sys

# Remove the module from cache if it exists
if 'src.labeler.narrative_labeler' in sys.modules:
    del sys.modules['src.labeler.narrative_labeler']

# Force reimport
from src.labeler.narrative_labeler import NarrativeLabeler, NarrativeSummarizerFlex
importlib.reload(sys.modules['src.labeler.narrative_labeler'])

import importlib
from src.prompts import labeler
importlib.reload(labeler)
from src.prompts.labeler import get_narrative_system_prompt
from src.prompts import labeler
from src.labeler.narrative_labeler import NarrativeLabeler, NarrativeSummarizerFlex

narrative_summarizer = NarrativeSummarizerFlex(llm_model="openai::gpt-4o")

# Sort and initialize columns once
    
# Get final recap
final_summary = narrative_summarizer.get_summaries(
    main_theme="Introduction of a high fee on H-1B visas",
    df=entities_final_summary,
    mode="final_summary_general_report",
)

DEBUG: FINAL SUMMARY GENERAL REPORT SYSTEM PROMPT

Forget all previous prompts.
You are assisting a professional analyst in creating a comprehensive final recap summary from summaries focused on the theme of 'Introduction of a high fee on H-1B visas'. Each daily summary is compiled from news articles that reference specific companies or individuals based on their strategic positioning, actions taken, or relationship to the main theme.
You have to create a final narrative summary that captures the most significant developments and insights from the summaries.

You are given a series of the following::
    - Name of the company or individual
    - Summaries about the theme of 'Introduction of a high fee on H-1B visas' about the company or the individual

1. **Task**: 
    - You must create a final narrative summary that captures the most significant developments and insights from the summaries.
    - Do not include any information that is not related to the theme of 'Introduction of a hi

Querying OpenAI...:   0%|          | 0/1 [00:00<?, ?it/s]

Querying OpenAI...: 100%|██████████| 1/1 [00:08<00:00,  8.87s/it]


In [137]:
final_summary

'{"summary": "The introduction of a $100,000 annual fee on H-1B visas has significantly impacted various sectors, particularly the tech industry, which heavily relies on foreign talent. Major companies like Alphabet Inc., Amazon.com Inc., Microsoft Corp., and IBM Corp. Pty. Ltd. are facing increased operational costs and potential talent shortages. This fee is expected to double hiring costs, prompting these companies to reassess their human resources strategies, potentially slowing innovation and delaying projects in critical areas such as artificial intelligence and cloud computing. The policy, introduced by Donald Trump, aims to curb perceived abuses of the H-1B program and protect American jobs, but it has led to confusion and apprehension among tech companies, with fears of workforce disruption and potential downsizing.\\n\\nIndian IT firms like Wipro Ltd., Infosys Ltd., and Tata Consultancy Services Ltd. are particularly affected, as they rely heavily on H-1B visas for their oper

In [128]:
entities_final_summary.columns

Index(['Entity', 'summary', 'key_points', 'quotes', 'country', 'quote_count'], dtype='object')

In [126]:
entities_final_summary

,Entity,summary,key_points,quotes,country,quote_count
0,Alphabet Inc.,"The introduction of a high fee on H-1B visas, ...",[],[],NaN,7
1,Donald Trump,"In September 2025, Donald Trump announced a si...",[],[],NaN,7
2,Wipro Ltd.,Wipro Ltd. is navigating significant challenge...,[],[],NaN,7
3,Amazon.com Inc.,Amazon.com Inc. is significantly impacted by t...,[],[],NaN,7
4,Microsoft Corp.,"The introduction of a $100,000 fee on H-1B vis...",[],[],NaN,7
5,Infosys Ltd.,Infosys Ltd. is navigating significant challen...,[],[],NaN,7
6,IBM Corp. Pty. Ltd.,IBM Corp. Pty. Ltd. is navigating a challengin...,[],[],NaN,6
7,Thomson Reuters Corp.,"The introduction of a $100,000 annual fee on H...",[],[],NaN,6
8,Tata Consultancy Services Ltd.,Tata Consultancy Services Ltd. (TCS) is signif...,[],[],NaN,5
9,Dewang,"Dewang, led by Founder & CEO Dewang Neralla, h...",[],[],NaN,1


In [151]:
import importlib
import sys

# Rimuovi completamente il modulo dalla cache
if 'src.report.html' in sys.modules:
    del sys.modules['src.report.html']

# Reimporta da zero
from src.report.html import generate_interactive_timeline_dashboard_html


generate_interactive_timeline_dashboard_html(
    df_highlights=df_highlights,
    df_company_summaries=df_company_summaries,
    plotly_fig=fig_enhanced,
    countries_dict=countries_dict,
    df_entities_final_summary=entities_final_summary,
    company_stats=company_stats,
    unique_sentences_count=unique_sentences_count,
    df_summaries_entities=df_summaries_entities,
    final_summary=final_summary,  # ← PRENDI SOLO IL VALORE
    output_file="full_dashboard_with_people_and_summary.html"
)

Interactive timeline dashboard generated: full_dashboard_with_people_and_summary.html


'full_dashboard_with_people_and_summary.html'